In [ ]:
# --- ensure repo-root cwd so data/ paths resolve from anywhere ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# State of Health (SoH) for a single VIN

This notebook computes a clean SoH curve for **one vehicle** from the data extracted into
`data/extracted/` by `import_data.ipynb`.

**Why we use the BMS-reported `batterySoh` field rather than deriving SoH from capacity:**
a capacity-based SoH (full-charge capacity ÷ nominal capacity) requires pack **current / Ah
throughput** for coulomb counting. The telemetry schema here has `batterySoc`, `batteryVoltage`,
`batteryTemperature`, and `odometer` — but **no current channel** — so an independent capacity
estimate isn't reliable. Instead we take the BMS `batterySoh`, clean its noise, and turn it into
a robust health-vs-time / health-vs-distance curve plus a degradation rate.

The notebook runs against whatever has been extracted so far — it auto-selects the VIN with the
most rows currently available (override `VIN` in the config cell to pick a specific vehicle).

## 1. Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
EXTRACTED = DATA_DIR / "extracted"
SOH_OUT = DATA_DIR / "soh"
SOH_OUT.mkdir(parents=True, exist_ok=True)

# Plausible bounds for a reported SoH percentage. Values outside are treated as garbage.
SOH_MIN, SOH_MAX = 1.0, 100.0

# Override to analyse a specific vehicle, e.g. VIN = "MD9EMHDL22E217035".
# Leave as None to auto-pick the VIN with the most extracted rows.
VIN = None

dataset = ds.dataset(EXTRACTED, format="parquet")
print(f"Reading from {EXTRACTED}  (files present: {len(dataset.files)})")

## 2. Pick the VIN

Counts rows per VIN across the extracted files and selects the one with the most data
(unless `VIN` was set manually above).

In [ ]:
vin_counts = (
    dataset.to_table(columns=["vin"])
    .column("vin")
    .to_pandas()
    .value_counts()

)
print("Rows available per VIN so far:")
print(vin_counts.to_string())

if VIN is None:
    VIN = vin_counts.index[0]
print(f"\nAnalysing VIN: {VIN}  ({vin_counts.get(VIN, 0):,} rows)")

## 3. Load and clean this VIN's SoH signal

Pushes a `vin == VIN` filter into the Parquet read (so only this vehicle's rows load),
parses the `eventAt` epoch-ms timestamp, coerces SoH/SoC/odometer to numeric, keeps only
physically plausible SoH values, and sorts by time.

In [ ]:
cols = ["eventAt", "vin", "batterySoh", "batterySoc", "odometer",
        "batteryVoltage", "batteryTemperature"]
tbl = dataset.to_table(columns=cols, filter=pc.field("vin") == VIN)
df = tbl.to_pandas()
print(f"Loaded {len(df):,} raw rows for {VIN}")

df["eventAt"] = pd.to_datetime(df["eventAt"], unit="ms", errors="coerce")
for c in ["batterySoh", "batterySoc", "odometer", "batteryVoltage", "batteryTemperature"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

clean = (
    df.dropna(subset=["eventAt", "batterySoh"])
      .query("@SOH_MIN <= batterySoh <= @SOH_MAX")
      .sort_values("eventAt")
      .reset_index(drop=True)

)
dropped = len(df) - len(clean)
print(f"Kept {len(clean):,} rows with valid SoH; dropped {dropped:,} (null/out-of-range).")
clean[["eventAt", "batterySoh", "batterySoc", "odometer"]].describe()

## 4. Daily SoH curve

Raw BMS SoH is noisy and reported many times per day. We take the **daily median** as a
robust health value, then a 7-day rolling median to show the underlying trend.

In [ ]:
daily = (
    clean.set_index("eventAt")["batterySoh"]
         .resample("1D").median()
         .dropna()
         .rename("soh_daily_median")
         .to_frame()

)
daily["soh_7d"] = daily["soh_daily_median"].rolling(7, min_periods=1).median()
# Attach the day's median odometer for the SoH-vs-distance view.
daily["odometer"] = (
    clean.set_index("eventAt")["odometer"].resample("1D").median().reindex(daily.index)

)
print(f"{len(daily):,} days of SoH, {daily.index.min().date()} → {daily.index.max().date()}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily.index, daily["soh_daily_median"], ".", ms=3, alpha=0.3, label="daily median")
ax.plot(daily.index, daily["soh_7d"], "-", lw=2, label="7-day median")
ax.set_title(f"SoH over time — {VIN}")
ax.set_ylabel("State of Health (%)"); ax.set_xlabel("Date")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 5. SoH vs odometer and degradation rate

Fits a linear trend of SoH against both time (per year) and distance (per 1000 km).
These slopes are the headline degradation rates and feed directly into RUL estimation.

In [ ]:
d = daily.dropna(subset=["odometer"]).copy()

# Time trend: %/year.
t_years = (d.index - d.index[0]).total_seconds() / (365.25 * 24 * 3600)
soh_per_year = np.polyfit(t_years, d["soh_daily_median"], 1)[0] if len(d) > 2 else np.nan

# Distance trend: %/1000 km (only meaningful if the odometer actually advances).
km = d["odometer"].to_numpy()
if len(d) > 2 and np.nanmax(km) > np.nanmin(km):
    soh_per_1000km = np.polyfit(km, d["soh_daily_median"], 1)[0] * 1000
else:
    soh_per_1000km = np.nan

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(d["odometer"], d["soh_daily_median"], s=8, alpha=0.4)
ax.set_title(f"SoH vs odometer — {VIN}")
ax.set_xlabel("Odometer (km)"); ax.set_ylabel("State of Health (%)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Degradation vs time:     {soh_per_year:.2f} %/year")
print(f"Degradation vs distance: {soh_per_1000km:.3f} %/1000 km")

## 6. Summary and save

Reports current SoH (recent 7-day median) and total drop, then writes the daily SoH
series to `data/soh/<VIN>_soh_daily.parquet` for downstream RUL work.

In [ ]:
soh_start = daily["soh_7d"].iloc[0]
soh_now = daily["soh_7d"].iloc[-1]
summary = {
    "vin": VIN,
    "rows_used": int(len(clean)),
    "days": int(len(daily)),
    "date_start": str(daily.index.min().date()),
    "date_end": str(daily.index.max().date()),
    "soh_start_pct": round(float(soh_start), 2),
    "soh_current_pct": round(float(soh_now), 2),
    "soh_drop_pct": round(float(soh_start - soh_now), 2),
    "degradation_pct_per_year": round(float(soh_per_year), 3),
    "degradation_pct_per_1000km": round(float(soh_per_1000km), 4),
}
for k, v in summary.items():
    print(f"{k:>28}: {v}")

out = SOH_OUT / f"{VIN}_soh_daily.parquet"
daily.reset_index().to_parquet(out, index=False)
print(f"\nSaved daily SoH series → {out}")